In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import polars as pl
import pyarrow
import gc
from itertools import combinations
import matplotlib.pyplot as plt
sys.path.append(str(Path("..")))
# Backtest dataset construction - universe tradable daily 
from src.ingest.ingest_backtest import build_backtest_warmup_parquets
from src.universe.pair_universe import (build_universe_metadata_from_symbol_parquets,apply_sector_fixes,generate_candidate_pairs,
                                        build_daily_tradable_universe,)

Within same-sector Nasdaq-100 stocks, larger-cap or more liquid names may incorporate information earlier. If this is true, lagged returns of the leader should have explanatory power over future returns of the follower.



In [2]:
BACKTEST_DATA_DIR = Path("../data/backtest_1min_with_warmup_by_symbol")

SECTOR_FIXES = {"ASML": "SEMICONDUCTORS & RELATED DEVICES",
                "CCEP": "BEVERAGES",
                "PDD": "RETAIL-CATALOG & MAIL-ORDER HOUSES",
                "TRI": "SERVICES-BUSINESS SERVICES, NEC",}

# 1. Build Universe Metadata
universe_df = build_universe_metadata_from_symbol_parquets(BACKTEST_DATA_DIR)
universe_df = apply_sector_fixes(universe_df,SECTOR_FIXES)
# 2. Generate Same-Sector Candidate Pairs
df_candidate_pairs = generate_candidate_pairs(universe_df)

In [3]:
df_candidate_pairs

,stock_a,stock_b,sector
0,CCEP,KDP,BEVERAGES
1,CCEP,PEP,BEVERAGES
2,KDP,PEP,BEVERAGES
3,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)"
4,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES
...,...,...,...
185,SHOP,TEAM,SERVICES-PREPACKAGED SOFTWARE
186,SHOP,TTWO,SERVICES-PREPACKAGED SOFTWARE
187,SNPS,TEAM,SERVICES-PREPACKAGED SOFTWARE
188,SNPS,TTWO,SERVICES-PREPACKAGED SOFTWARE


In [ ]:
adi = pd.read_parquet("../data/backtest_1min_with_warmup_by_symbol/ADI_backtest_1min_warmup.parquet")
amat = pd.read_parquet("../data/backtest_1min_with_warmup_by_symbol/AMAT_backtest_1min_warmup.parquet")
amd = pd.read_parquet("../data/backtest_1min_with_warmup_by_symbol/AMD_backtest_1min_warmup.parquet")

# test
def add_log_return(df):
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    return df
dfs = { 'adi' : adi,'amat' : amat, 'amd': amd}
for symbol, df in dfs.items():
    dfs[symbol] = add_log_return(df)
# two candidate merge 
adi_amat = pd.merge(adi[['date', 'log_return']], amat[['date', 'log_return']], on='date', suffixes=('adi', 'amat'))
# adi & amd
adi_amd = pd.merge(adi[['date', 'log_return']], amd[['date', 'log_return']], on='date', suffixes=('adi', 'amd'))
pairs = {"ADI_AMAT": adi_amat,"ADI_AMD": adi_amd}
lags = [1,5,15,30]
results = []
for pair_name, df in pairs.items():
    col1 = [c for c in df.columns if "adi" in c][0]
    col2 = [c for c in df.columns if col1 != c and "log_return" in c][0]
    for lag in lags:
        corr_forward = (df[col1].corr(df[col2].shift(-lag)))
        corr_reverse = (df[col2].corr(df[col1].shift(-lag)))
        results.append([pair_name,lag,corr_forward,corr_reverse])
results = pd.DataFrame(results,columns=["pair","lag","A_to_B","B_to_A"])
results["lead_score"] = (results["A_to_B"]- results["B_to_A"])
results

,pair,lag,A_to_B,B_to_A,lead_score
0,ADI_AMAT,1,0.004685,0.020605,-0.015920
1,ADI_AMAT,5,-0.003520,-0.003696,0.000176
2,ADI_AMAT,15,0.007188,0.010060,-0.002873
3,ADI_AMAT,30,0.000296,0.002476,-0.002180
4,ADI_AMD,1,0.005376,0.019947,-0.014571
5,ADI_AMD,5,-0.000682,-0.002218,0.001536
6,ADI_AMD,15,0.000774,0.003551,-0.002777
7,ADI_AMD,30,-0.000906,0.003449,-0.004355


In [5]:
from pathlib import Path

DATA_DIR = Path("../data/backtest_1min_with_warmup_by_symbol")
LAGS = [1, 5, 15, 30]

def load_symbol_returns(symbol, data_dir=DATA_DIR):
    path = data_dir / f"{symbol}_backtest_1min_warmup.parquet"
    df = pd.read_parquet(path)
    df = df[["date", "close"]].copy()
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    return df[["date", "log_return"]].dropna()


def compute_pair_lead_lag(stock_a, stock_b, sector=None, lags=LAGS):
    a = load_symbol_returns(stock_a).rename(columns={"log_return": "r_a"})
    b = load_symbol_returns(stock_b).rename(columns={"log_return": "r_b"})
    pair_df = pd.merge(a, b, on="date", how="inner").dropna()
    rows = []
    for lag in lags:
        a_to_b = pair_df["r_a"].corr(pair_df["r_b"].shift(-lag))
        b_to_a = pair_df["r_b"].corr(pair_df["r_a"].shift(-lag))
        lead_score = a_to_b - b_to_a
        rows.append({
            "stock_a": stock_a,
            "stock_b": stock_b,
            "sector": sector,
            "lag": lag,
            "A_to_B": a_to_b,
            "B_to_A": b_to_a,
            "lead_score": lead_score,
            "abs_lead_score": abs(lead_score),
            "empirical_leader": stock_a if lead_score > 0 else stock_b,
            "empirical_follower": stock_b if lead_score > 0 else stock_a,
            "n_obs": len(pair_df)
        })
    return pd.DataFrame(rows)

all_results = []

for _, row in df_candidate_pairs.iterrows():
    pair_result = compute_pair_lead_lag(
        stock_a=row["stock_a"],
        stock_b=row["stock_b"],
        sector=row["sector"],
        lags=LAGS
    )
    all_results.append(pair_result)

lead_lag_results = pd.concat(all_results, ignore_index=True)
lead_lag_clean = lead_lag_results.query("n_obs >= 10000").copy()
lead_lag_clean.sort_values("abs_lead_score", ascending=False).head(20)

,stock_a,stock_b,sector,lag,A_to_B,B_to_A,lead_score,abs_lead_score,empirical_leader,empirical_follower,n_obs
246,AVGO,MPWR,SEMICONDUCTORS & RELATED DEVICES,15,0.000449,-0.086163,0.086613,0.086613,AVGO,MPWR,12991
408,MELI,PYPL,"SERVICES-BUSINESS SERVICES, NEC",1,0.005223,0.065919,-0.060696,0.060696,PYPL,MELI,477455
708,MSFT,SNPS,SERVICES-PREPACKAGED SOFTWARE,1,0.054634,0.000429,0.054206,0.054206,MSFT,SNPS,477455
334,MPWR,NXPI,SEMICONDUCTORS & RELATED DEVICES,15,0.042501,-0.008463,0.050965,0.050965,MPWR,NXPI,12991
138,AMAT,MPWR,SEMICONDUCTORS & RELATED DEVICES,15,0.011290,-0.039127,0.050417,0.050417,AMAT,MPWR,12991
676,INTU,MSFT,SERVICES-PREPACKAGED SOFTWARE,1,-0.001523,0.048300,-0.049824,0.049824,MSFT,INTU,477455
54,INSM,VRTX,PHARMACEUTICAL PREPARATIONS,15,0.045382,-0.003079,0.048461,0.048461,INSM,VRTX,12991
224,ASML,NVDA,SEMICONDUCTORS & RELATED DEVICES,1,0.003750,0.049794,-0.046044,0.046044,NVDA,ASML,477455
160,AMD,ASML,SEMICONDUCTORS & RELATED DEVICES,1,0.046654,0.002959,0.043696,0.043696,AMD,ASML,477455
16,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES,1,0.008401,0.051989,-0.043587,0.043587,CMCSA,CHTR,477455


In [8]:
lead_lag_clean["n_obs"].describe()

count       708.000000
mean     389031.870056
std      158986.032952
min       12991.000000
25%      416961.000000
50%      477455.000000
75%      477455.000000
max      477455.000000
Name: n_obs, dtype: float64

In [9]:
lead_lag_clean.groupby("lag")["n_obs"].median()

lag
1     477455.0
5     477455.0
15    477455.0
30    477455.0
Name: n_obs, dtype: float64

In [10]:
lead_lag_clean.groupby("empirical_leader").size()

empirical_leader
ADBE     22
ADI      16
ADP       5
ADSK     11
ALNY      9
AMAT     21
AMD      36
AMGN      2
AMZN      2
APP       5
ASML     16
AVGO     21
CCEP      3
CDNS     21
CHTR      2
CMCSA     2
CRWD     31
CSGP      6
CTSH      2
DASH     11
DDOG     31
EA       27
EXC       2
FTNT      2
GILD      2
GOOG      4
GOOGL     3
INSM      5
INTC     33
INTU     13
KDP       5
MCHP     18
MELI      4
MPWR     19
MRVL     26
MSFT     34
MU       29
NVDA     33
NXPI     16
PANW      2
PDD       2
PEP       4
PLTR     35
PYPL     13
REGN      5
SNPS     11
STX       2
TEAM     19
TRI       6
TSLA      4
TTWO      9
TXN      28
VRSK      4
VRTX      5
WDAY      3
WDC       2
XEL       2
ZS        2
dtype: int64

In [11]:
lead_lag_clean.groupby(["sector","empirical_leader"]).size()

sector                                                empirical_leader
BEVERAGES                                             CCEP                 3
                                                      KDP                  5
                                                      PEP                  4
BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)       AMGN                 2
                                                      GILD                 2
CABLE & OTHER PAY TELEVISION SERVICES                 CHTR                 2
                                                      CMCSA                2
COMPUTER PERIPHERAL EQUIPMENT, NEC                    FTNT                 2
                                                      PANW                 2
COMPUTER STORAGE DEVICES                              STX                  2
                                                      WDC                  2
ELECTRIC & OTHER SERVICES COMBINED                    EXC                  2
     

In [ ]:
leader_counts = (lead_lag_clean["empirical_leader"].value_counts().reset_index())
leader_counts.columns = ["symbol", "count"]

# 1. Crear el gráfico de barras ordenado de mayor a menor
fig = px.bar(
    leader_counts,
    x="symbol",
    y="count",
    title="Frecuencia de Liderazgo por Acción (Efecto Lead-Lag)",
    labels={"symbol": "Acción (Ticker)", "count": "Veces como Líder"},
    text_auto=True,  # Muestra el número exacto sobre cada barra
    color="count",  # Aplica un degradado de color según el conteo
    color_continuous_scale="Viridis",
)

# 2. Mejorar el diseño para que sea fácil de leer
fig.update_layout(
    xaxis_tickangle=-45,  # Inclina los tickers si son muchos (tienes 95)
    xaxis={"categoryorder": "total descending"},  # Asegura el orden descendente
    plot_bgcolor="white",  # Fondo limpio
    height=600,  # Altura ajustable para acomodar los 95 símbolos
)

# 3. Mostrar el gráfico
fig.show()


In [14]:
lead_lag_clean.groupby("lag")["abs_lead_score"].describe()

,count,mean,std,min,25%,50%,75%,max
lag,,,,,,,,
1,177.0,0.015758,0.012543,0.000104,0.005185,0.013563,0.024017,0.060696
5,177.0,0.002786,0.002386,0.000038,0.001021,0.002299,0.003789,0.014330
15,177.0,0.007978,0.010304,0.000086,0.002665,0.005576,0.008994,0.086613
30,177.0,0.003540,0.003522,0.000048,0.001371,0.002637,0.004573,0.025752


In [15]:
lag_1 = lead_lag_clean.query("lag == 1").copy()
lag_1["empirical_leader"].value_counts()

empirical_leader
AMD      12
NVDA     11
INTC     10
MSFT     10
PLTR     10
MRVL      9
DDOG      9
MU        8
CRWD      8
AMAT      7
ADBE      7
EA        7
TXN       6
MCHP      5
TEAM      5
ADI       4
PYPL      4
CDNS      4
AVGO      3
DASH      3
TTWO      3
PEP       2
INSM      2
REGN      2
VRTX      2
NXPI      2
CSGP      2
ADP       2
GOOGL     2
INTU      2
KDP       1
GILD      1
CMCSA     1
FTNT      1
STX       1
EXC       1
TSLA      1
PDD       1
ASML      1
TRI       1
WDAY      1
CTSH      1
GOOG      1
ADSK      1
Name: count, dtype: int64

Sin avanzar más:

La señal parece concentrarse en lag=1 minuto.
La distribución de líderes no parece uniforme.
Aparecen nombres repetidamente (AMD, NVDA, INTC, MSFT, PLTR).
La métrica está produciendo una estructura interpretable, no puro ruido.


--- 
# Validación estadística del efecto lead-lag

In [36]:
msft= pd.read_parquet("../data/backtest_1min_with_warmup_by_symbol/AMD_backtest_1min_warmup.parquet")
snps = pd.read_parquet("../data/backtest_1min_with_warmup_by_symbol/NVDA_backtest_1min_warmup.parquet")

def add_log_return(df):
    # Fórmula correcta: ln( Price_t / Price_t-1 )
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    return df
add_log_return(snps)
add_log_return(msft)
amd_nvda = pd.merge(msft[['date', 'log_return']], snps[['date', 'log_return']], on='date', suffixes=('_msft', '_snps'))
amd_nvda


,date,log_return_msft,log_return_snps
0,2021-06-01 09:15:00,NaN,NaN
1,2021-06-01 09:16:00,-0.000370,-0.000153
2,2021-06-01 09:17:00,0.000123,0.000000
3,2021-06-01 09:18:00,0.000246,-0.000381
4,2021-06-01 09:19:00,0.000000,-0.000387
...,...,...,...
477451,2026-02-04 15:56:00,-0.001586,-0.000946
477452,2026-02-04 15:57:00,-0.003977,0.000287
477453,2026-02-04 15:58:00,-0.002643,0.001175
477454,2026-02-04 15:59:00,-0.000249,-0.002380


In [37]:
def lead_score(x, y, lag=1):
    a_to_b = x.corr(y.shift(-lag))
    b_to_a = y.corr(x.shift(-lag))
    return a_to_b - b_to_a

def permutation_test_lead_score(x,y,lag=1,n_permutations=1000,random_state=42):
    rng = np.random.default_rng(random_state)
    real_score = lead_score(x, y, lag)
    random_scores = []
    y_clean = y.dropna().values
    for _ in range(n_permutations):
        y_perm = rng.permutation(y_clean)
        y_perm = pd.Series(
            y_perm,
            index=y.dropna().index)
        score = lead_score( x.loc[y_perm.index],y_perm,lag)
        random_scores.append(score)
    random_scores = np.array(random_scores)
    p_value = np.mean( np.abs(random_scores)>=np.abs(real_score))

    return {"real_score": real_score,"random_mean": random_scores.mean(),
            "random_std": random_scores.std(),"p_value": p_value,"random_scores": random_scores}

result = permutation_test_lead_score(amd_nvda["log_return_msft"],amd_nvda["log_return_snps"],lag=1)
result

{'real_score': np.float64(0.0021254219370133824),
 'random_mean': np.float64(-7.5391717447962e-05),
 'random_std': np.float64(0.002062638069602675),
 'p_value': np.float64(0.274),
 'random_scores': array([-1.49050680e-04,  9.85028445e-04,  9.92420121e-04, -1.61419194e-03,
         2.24718885e-03,  4.57800460e-03, -2.83024129e-04, -1.08426814e-03,
         2.82579664e-03, -2.22318374e-03,  2.13307782e-03, -1.32445827e-03,
         2.03850661e-03, -1.19478779e-03, -1.05027677e-03, -3.58873074e-03,
         5.59167053e-05, -1.87468517e-03, -1.31217980e-03,  6.12621021e-04,
        -5.48977393e-04, -1.26988076e-03,  8.18064793e-04, -1.43082825e-03,
        -2.22579859e-05, -2.42785220e-03, -2.59706548e-03,  2.76190673e-03,
        -1.25535223e-03, -1.63318954e-03,  2.86500196e-03,  1.83295061e-03,
         8.75759004e-04,  1.12937520e-03, -3.11822931e-03,  2.10953801e-03,
        -7.97503387e-04, -2.14091585e-04,  3.54559918e-03,  1.64235055e-05,
        -2.80381448e-03, -6.44624299e-04,  

In [38]:
import plotly.express as px
fig = px.histogram(x=result["random_scores"],nbins=50,title="Permutation Distribution of Lead Score")
fig.add_vline(x=result["real_score"],line_width=3)
fig.show()

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/backtest_1min_with_warmup_by_symbol")

def load_symbol_returns(symbol, data_dir=DATA_DIR):
    path = data_dir / f"{symbol}_backtest_1min_warmup.parquet"
    
    df = pd.read_parquet(path)
    df = df[["date", "close"]].copy()
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    
    return df[["date", "log_return"]].dropna()


# Cargar retornos solo de los símbolos que aparecen en df_candidate_pairs
symbols = sorted(
    set(df_candidate_pairs["stock_a"])
    .union(set(df_candidate_pairs["stock_b"]))
)

returns_by_symbol = {}

for symbol in symbols:
    returns_by_symbol[symbol] = load_symbol_returns(symbol)

len(returns_by_symbol)

62

In [12]:
def lead_score(x, y, lag=1):
    a_to_b = x.corr(y.shift(-lag))
    b_to_a = y.corr(x.shift(-lag))
    return a_to_b - b_to_a


def permutation_test_lead_score(
    x,
    y,
    lag=1,
    n_permutations=1000,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    real_score = lead_score(x, y, lag)

    random_scores = []

    y_clean = y.dropna()
    x_aligned = x.loc[y_clean.index]

    for _ in range(n_permutations):
        y_perm = rng.permutation(y_clean.values)

        y_perm = pd.Series(
            y_perm,
            index=y_clean.index
        )

        score = lead_score(x_aligned, y_perm, lag)
        random_scores.append(score)

    random_scores = np.array(random_scores)

    p_value = np.mean(
        np.abs(random_scores) >= np.abs(real_score)
    )

    return {
        "lead_score": real_score,
        "random_mean": random_scores.mean(),
        "random_std": random_scores.std(),
        "p_value": p_value
    }


MIN_OBS = 10000
N_PERMUTATIONS = 1000
LAG = 1

all_tests = []

for _, row in df_candidate_pairs.head(40).iterrows():

    stock_a = row["stock_a"]
    stock_b = row["stock_b"]
    sector = row["sector"]

    a = returns_by_symbol[stock_a].rename(columns={"log_return": "r_a"})
    b = returns_by_symbol[stock_b].rename(columns={"log_return": "r_b"})

    pair_df = pd.merge(
        a,
        b,
        on="date",
        how="inner"
    ).dropna()

    n_obs = len(pair_df)

    if n_obs < MIN_OBS:
        print(f"Skipping {stock_a}-{stock_b}: only {n_obs} observations")
        continue

    test = permutation_test_lead_score(
        pair_df["r_a"],
        pair_df["r_b"],
        lag=LAG,
        n_permutations=N_PERMUTATIONS,
        random_state=42
    )

    lead_score_value = test["lead_score"]

    all_tests.append({
        "pair": f"{stock_a}-{stock_b}",
        "stock_a": stock_a,
        "stock_b": stock_b,
        "sector": sector,
        "lag": LAG,
        "lead_score": lead_score_value,
        "abs_lead_score": abs(lead_score_value),
        "p_value": test["p_value"],
        "random_mean": test["random_mean"],
        "random_std": test["random_std"],
        "empirical_leader": stock_a if lead_score_value > 0 else stock_b,
        "empirical_follower": stock_b if lead_score_value > 0 else stock_a,
        "n_obs": n_obs
    })

lead_lag_significance = pd.DataFrame(all_tests)

lead_lag_significance.sort_values("p_value").head(20)

Skipping COST-WMT: only 5683 observations


,pair,stock_a,stock_b,sector,lag,lead_score,abs_lead_score,p_value,random_mean,random_std,empirical_leader,empirical_follower,n_obs
0,CCEP-KDP,CCEP,KDP,BEVERAGES,1,-0.017671,0.017671,0.0,-0.000032,0.002375,KDP,CCEP,315461
1,CCEP-PEP,CCEP,PEP,BEVERAGES,1,-0.027135,0.027135,0.0,0.000017,0.002475,PEP,CCEP,315461
2,KDP-PEP,KDP,PEP,BEVERAGES,1,-0.009183,0.009183,0.0,-0.000020,0.002031,PEP,KDP,477455
3,AMGN-GILD,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)",1,-0.021105,0.021105,0.0,-0.000020,0.001951,GILD,AMGN,477455
4,CHTR-CMCSA,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES,1,-0.043587,0.043587,0.0,-0.000013,0.002097,CMCSA,CHTR,477455
14,REGN-VRTX,REGN,VRTX,PHARMACEUTICAL PREPARATIONS,1,-0.031920,0.031920,0.0,0.000036,0.001955,VRTX,REGN,477455
11,ALNY-VRTX,ALNY,VRTX,PHARMACEUTICAL PREPARATIONS,1,-0.026512,0.026512,0.0,-0.000119,0.003241,VRTX,ALNY,172143
8,PCAR-TSLA,PCAR,TSLA,MOTOR VEHICLES & PASSENGER CAR BODIES,1,-0.008360,0.008360,0.0,-0.000024,0.002049,TSLA,PCAR,477455
24,ADI-MU,ADI,MU,SEMICONDUCTORS & RELATED DEVICES,1,-0.014484,0.014484,0.0,-0.000068,0.002054,MU,ADI,477455
25,ADI-NVDA,ADI,NVDA,SEMICONDUCTORS & RELATED DEVICES,1,-0.017401,0.017401,0.0,-0.000070,0.001989,NVDA,ADI,477455


In [13]:
import plotly.express as px

fig = px.scatter(
    lead_lag_significance,
    x="abs_lead_score",
    y="p_value",
    hover_data=["pair","sector"],
    title="Lead Score vs Statistical Significance"
)

fig.add_hline(
    y=0.05,
    line_dash="dash"
)

fig.show()

In [14]:
lead_lag_significance.groupby("empirical_leader").size().sort_values(ascending=False)

empirical_leader
AMAT     7
ADI      4
AMD      2
INSM     2
INTC     2
VRTX     2
REGN     2
MRVL     2
MU       2
PEP      2
NVDA     2
EXC      1
CMCSA    1
KDP      1
MCHP     1
GILD     1
FTNT     1
PDD      1
STX      1
TSLA     1
TXN      1
dtype: int64

Hasta ahora encontramos:

- La señal vive principalmente en lag=1.
- Algunos pares no tienen efecto significativo.
- Otros pares sí tienen lead_score muy por encima de lo esperado bajo permutación.
-El líder empírico no necesariamente coincide con una narrativa simple de market cap.
- ADI, por ejemplo, aparece muchas veces como follower frente a AMD, AMAT, INTC, MRVL, MU, NVDA, TXN.

# ¿Qué tan buena era la regla de market cap para identificar al líder?

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/backtest_1min_with_warmup_by_symbol")

def get_market_cap(symbol, data_dir=DATA_DIR):
    path = data_dir / f"{symbol}_backtest_1min_warmup.parquet"
    df = pd.read_parquet(path, columns=["market_cap"])
    return df["market_cap"].dropna().iloc[0]


def get_market_cap_leader(stock_a, stock_b):
    market_cap_a = get_market_cap(stock_a)
    market_cap_b = get_market_cap(stock_b)

    if market_cap_a >= market_cap_b:
        return stock_a, stock_b, market_cap_a, market_cap_b
    else:
        return stock_b, stock_a, market_cap_a, market_cap_b


rows = []

for _, row in lead_lag_significance.iterrows():

    stock_a = row["stock_a"]
    stock_b = row["stock_b"]

    market_cap_leader, market_cap_follower, market_cap_a, market_cap_b = get_market_cap_leader(
        stock_a,
        stock_b
    )

    rows.append({
        "pair": row["pair"],
        "stock_a": stock_a,
        "stock_b": stock_b,
        "sector": row["sector"],
        "lag": row["lag"],
        "lead_score": row["lead_score"],
        "abs_lead_score": row["abs_lead_score"],
        "p_value": row["p_value"],
        "empirical_leader": row["empirical_leader"],
        "empirical_follower": row["empirical_follower"],
        "market_cap_leader": market_cap_leader,
        "market_cap_follower": market_cap_follower,
        "market_cap_a": market_cap_a,
        "market_cap_b": market_cap_b,
        "leader_match": row["empirical_leader"] == market_cap_leader,
        "n_obs": row["n_obs"]
    })

leader_comparison = pd.DataFrame(rows)

leader_comparison.head()

,pair,stock_a,stock_b,sector,lag,lead_score,abs_lead_score,p_value,empirical_leader,empirical_follower,market_cap_leader,market_cap_follower,market_cap_a,market_cap_b,leader_match,n_obs
0,CCEP-KDP,CCEP,KDP,BEVERAGES,1,-0.017671,0.017671,0.0,KDP,CCEP,CCEP,KDP,4.254597e+10,3.865169e+10,False,315461
1,CCEP-PEP,CCEP,PEP,BEVERAGES,1,-0.027135,0.027135,0.0,PEP,CCEP,PEP,CCEP,4.254597e+10,2.271097e+11,True,315461
2,KDP-PEP,KDP,PEP,BEVERAGES,1,-0.009183,0.009183,0.0,PEP,KDP,PEP,KDP,3.865169e+10,2.271097e+11,True,477455
3,AMGN-GILD,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)",1,-0.021105,0.021105,0.0,GILD,AMGN,AMGN,GILD,1.973818e+11,1.814246e+11,False,477455
4,CHTR-CMCSA,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES,1,-0.043587,0.043587,0.0,CMCSA,CHTR,CMCSA,CHTR,2.838826e+10,1.097343e+11,True,477455


In [19]:
leader_comparison["leader_match"].mean()

np.float64(0.6153846153846154)

In [20]:
leader_comparison.query("p_value < 0.05")["leader_match"].mean()

np.float64(0.6785714285714286)

El liderazgo basado en market capitalization no es arbitrario. Aproximadamente 68% de los pares con evidencia estadística de lead-lag presentan coincidencia entre el líder empírico y el líder definido por market capitalization. Esto sugiere que market cap captura una parte importante del proceso de difusión de información, aunque existe una fracción relevante de pares (~32%) donde el liderazgo observado difiere del esperado, abriendo la posibilidad de construir reglas de selección más informativas que la simple jerarquía por tamaño.